# Scenario Playground Step 13: All-Target Ridge Evaluation

Objective: apply the frozen Step 12 full-history Ridge methodology to all 12 targets, compare every target with three naive baselines over five purged walk-forward folds, and classify each target for later research inference.

Scope: this notebook reads only the 464 retained pre-final-test rows of the frozen 584-row, 58-feature event-aware matrix. It does not parse or evaluate the 114 locked final-test targets, export model binaries, run Step 14, or touch application code.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "scenario_modeling").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Run this notebook from inside the InsightPulse repository.")
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scenario_modeling.src.all_target_models import (
    build_all_target_artifacts,
    persist_all_target_artifacts,
)

artifacts = build_all_target_artifacts()
output_paths = persist_all_target_artifacts(artifacts)
summary = artifacts["summary"]
{
    "python_executable": sys.executable,
    "training_gate": summary["training_gate"],
    "target_count": summary["target_count"],
    "classification_counts": summary["classification_counts"],
    "final_test_access_count": summary["matrix_guard"]["final_test_access_count"],
    "output_count": len(output_paths),
}


## Frozen folds, preprocessing scope, and final-test lock

Every target uses the same five expanding folds approved in Step 12. A row is eligible for training only when its `target_end_date` is earlier than the first target-baseline session in the next validation fold. Alpha selection and all preprocessing occur inside each purged training fold.

In [ ]:
folds = artifacts["fold_manifest"]
display(folds)
display(pd.Series(summary["matrix_guard"], name="value").to_frame())
assert folds["training_rows"].tolist() == [160, 222, 280, 342, 401]
assert folds["outer_purge_rows"].tolist() == [4, 2, 4, 2, 3]
assert summary["pretest_rows_evaluated"] == 464
assert summary["feature_count"] == 58
assert summary["matrix_guard"]["locked_final_test_target_rows_parsed"] == 0
assert summary["matrix_guard"]["final_test_access_count"] == 0


## All-target out-of-sample assessment

Each target has 300 Ridge predictions and 300 predictions from each naive strategy. Price targets are evaluated in percentage points; the 10-year yield is evaluated in basis points. A target is not approved merely because Ridge fits: it must pass predictive, magnitude, and scenario-response gates.

In [ ]:
assessment_columns = [
    "target", "unit", "final_pretest_alpha", "ridge_mae", "ridge_rmse",
    "ridge_oos_r_squared", "ridge_directional_accuracy",
    "best_naive_mae_strategy", "best_naive_mae",
    "ridge_mae_improvement_vs_best_naive_pct",
    "ridge_fold_mae_wins_vs_best_naive", "classification", "fallback_strategy",
]
assessment = artifacts["target_assessment"]
display(assessment[assessment_columns].round(4))
display(assessment["classification"].value_counts().rename("targets").to_frame())
assert len(assessment) == 12
assert artifacts["fold_predictions"].groupby(["target", "strategy"]).size().eq(300).all()
assert not assessment["prediction_pathological"].any()


## Fold stability and market regimes

Pooled metrics can hide weak periods. The next tables expose each Ridge fold and its low-, medium-, and high-VIX errors. These diagnostics are evidence for confidence classifications, not a new tuning loop.

In [ ]:
ridge_fold_mae = artifacts["fold_metrics"].loc[
    artifacts["fold_metrics"]["strategy"].eq("ridge_full_history")
].pivot(index="target", columns="fold", values="mae")
display(ridge_fold_mae.round(4))

ridge_regimes = artifacts["regime_metrics"].loc[
    artifacts["regime_metrics"]["strategy"].eq("ridge_full_history")
].pivot(index="target", columns="vix_regime", values="mae")
display(ridge_regimes.round(4))


## Scenario-shock responsiveness and sign stability

For every fold, three representative market states are held fixed while each of the eight scenario inputs is moved across its observed training range. The reported responses are predictive associations, not causal estimates. Meaningful-response thresholds are 0.05 percentage points for price targets and 1 basis point for the yield target.

In [ ]:
responsiveness_columns = [
    "target", "contract_input", "folds_with_meaningful_response",
    "latest_fold_meaningful", "direction_consistent_when_meaningful",
    "fold_state_checks", "meaningful_checks", "all_sensible",
]
responsiveness = artifacts["sensitivity_summary"]
display(responsiveness[responsiveness_columns])

target_response_summary = assessment[[
    "target", "responsive_inputs_any_fold",
    "responsive_inputs_three_or_more_folds", "responsive_inputs_all_folds",
    "responsive_inputs_latest_fold",
    "directionally_consistent_responsive_inputs",
    "scenario_coefficient_sign_stable_all_folds",
    "classification",
]]
display(target_response_summary)
assert len(responsiveness) == 12 * 8
assert artifacts["sensitivity"]["finite_and_sensible"].all()


## Frozen research configurations and residual risks

The pretest refits below freeze a research configuration for the next step but deliberately do not serialize production models. Residual diagnostics retain the largest misses, including crisis and oil-futures outliers, so apparent average performance is not overstated.

In [ ]:
configurations = artifacts["final_configurations"]
display(configurations[[
    "target", "selected_alpha", "training_rows", "inner_training_rows",
    "inner_validation_rows", "inner_purge_rows", "inner_validation_mae",
    "model_binary_exported", "locked_final_test_used",
]].round(4))
display(artifacts["residual_summary"].round(4))
assert configurations["training_rows"].eq(464).all()
assert not configurations["model_binary_exported"].any()
assert not configurations["locked_final_test_used"].any()


## Step 13 decision

No target earns the strongest experimental classification. Three targets pass only the low-confidence gate; nine require their documented naive fallback because Ridge is not reliably better. Step 14 may implement gated research inference, but these results do not establish causality or production readiness.

In [ ]:
checks = pd.Series(summary["validations"], name="passed")
display(checks.to_frame())
assert checks.all()
assert summary["classification_counts"] == {
    "APPROVED_FOR_EXPERIMENTAL_INFERENCE": 0,
    "APPROVED_WITH_LOW_CONFIDENCE": 3,
    "NAIVE_FALLBACK_REQUIRED": 9,
    "NOT_APPROVED": 0,
}
assert summary["training_gate"] == "STEP_13_ALL_TARGET_RIDGE_EVALUATION_COMPLETE_READY_FOR_STEP_14_WITH_TARGET_GATES"
print(summary["final_test_decision"])
print(summary["step_14_readiness"])
print(summary["training_gate"])
